<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold3_of3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold 3 (OpenFold3)

Predict protein, RNA, DNA, and small-molecule structures using [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) with the freely available [OpenFold3](https://github.com/aqlaboratory/openfold) weights (Apache 2.0 — no commercial restrictions).

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**.

Just fill in the **Input sequence(s)** cell and run all. Proteins, nucleic acids and ligands all go in a single box; attention/XLA flags are chosen automatically for your runtime (T4, newer GPUs, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- The OpenFold3 Team (2025) [Github](https://github.com/aqlaboratory/openfold-3) - [doi:10.5281/zenodo.19001000](https://doi.org/10.5281/zenodo.19001000)

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · OF3 weights: AlQuraishi Lab (Apache 2.0)

In [ ]:
#@title Install dependencies (~3 mins)
%%time
import os, time, glob

use_af3_weights = True #@param {type:"boolean"}
#@markdown - **use_af3_weights**: leave OFF to use the free, open **OpenFold3** weights. Turn ON to
#@markdown   download the official **AlphaFold 3** weights directly - now public, no login or key required.
#@markdown   Subject to the AF3 weights terms of use.
USE_NATIVE = use_af3_weights
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.17 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
  os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.5/alphafold3_open-3.1.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
  base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/v3.1.5'
  #base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
  for script in ['run_alphafold', 'convert_of3_weights']:
    os.system(f'wget -q -nc {base}/{script}.py')
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights: official AlphaFold 3 (public download) or OpenFold3 (default, free) - both in background
if USE_NATIVE:
  if not os.path.isfile('NATIVE_WEIGHTS_DONE'):
    print('Downloading official AlphaFold 3 weights (public, no login required)...')
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"; touch NATIVE_WEIGHTS_DONE) &')
elif not os.path.isfile('WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  os.system('(aws s3 cp s3://openfold/staging/of3-p2-155k.pt . --no-sign-request; touch WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

# Wait for background jobs
for sentinel in (['NATIVE_WEIGHTS_DONE', 'DATA_DONE'] if USE_NATIVE else ['WEIGHTS_DONE', 'DATA_DONE']):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if USE_NATIVE and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

if not USE_NATIVE and not os.path.isfile('af3_converted_weights/of3_ported_weights.bin.zst'):
  print("converting openfold3 weights...")
  os.system("python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights")

print('Setup complete!  ' + ('Using official AlphaFold 3 weights.' if USE_NATIVE else 'Using OpenFold3 weights.'))

In [ ]:
#@title Input: batch jobs from FASTA
import re, os, json, hashlib
from google.colab import drive

drive_folder = 'AF3_batch_results' #@param {type:"string"}
#@markdown Each job's results (and the running `batch_summary.csv`) are copied here
#@markdown right after that job finishes, so nothing is lost on a runtime interrupt.

drive.mount('/content/drive')
DRIVE_DIR = f'/content/drive/MyDrive/{drive_folder}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Backing up to {DRIVE_DIR}/')

fasta_path = '/content/drive/MyDrive/combined_input.fasta' #@param {type:"string"}
#@markdown Multi-FASTA, one complex per record. Chains within a record use `:`
#@markdown (e.g. `Protein1:bait`). Jobname = everything before `OS=` in the header.

#@markdown ### Global additions (applied to every complex)
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = 'ATP:MG' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '3' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]

def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

# Reads a multi-record FASTA, tolerating wrapped sequence lines
def parse_multi_fasta(path):
  records, header, seq_lines = [], None, []
  def flush():
    if header is not None:
      records.append((header, ''.join(seq_lines)))
  with open(path) as f:
    for line in f:
      line = line.rstrip('\n')
      if line.startswith('>'):
        flush()
        header, seq_lines = line[1:].strip(), []
      elif line.strip():
        seq_lines.append(line.strip())
  flush()
  return records

# UniProt-style headers: keep everything before "OS=" as the human-readable name
def jobname_from_header(header, maxlen=60):
  base = header.split('OS=')[0].strip()
  base = re.sub(r'\s+', '_', base)
  base = re.sub(r'[^\w-]', '', base).lower()
  return (base[:maxlen] or 'job')

dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)

seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')

# Same per-job chain builder as the single-job notebook, factored into a function
def build_chains(prot_seqs):
  chains, prot_groups, idx = [], {}, 0
  for seq in prot_seqs:
    cid = CHAIN_IDS[idx]; idx += 1
    if seq in prot_groups:
      ent = prot_groups[seq]
      ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
      ent['id'] = ids + [cid]
    else:
      ent = {'id': cid, 'sequence': seq, 'templates': []}
      if msa_mode == 'single_sequence':
        ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
      prot_groups[seq] = ent
      chains.append({'protein': ent})
  for seq in rna_seqs:
    c = {'id': CHAIN_IDS[idx], 'sequence': seq}
    if msa_mode == 'single_sequence':
      c['unpairedMsa'] = f'>query\n{seq}\n'
    chains.append({'rna': c}); idx += 1
  for seq in dna_seqs:
    chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1
  for code in ccd_codes:
    chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1
  for smiles in smiles_strs:
    chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1
  return chains

def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')

INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = []  # one entry per FASTA record
for header, seq in parse_multi_fasta(fasta_path):
  prot_seqs = [e.upper() for e in split_entries(seq)]
  if not prot_seqs:
    continue
  chains = build_chains(prot_seqs)
  flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
  jobname = jobname_from_header(header) + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]
  job_dir, json_path = f'{OUTPUT_DIR}/{jobname}', f'{INPUT_DIR}/{jobname}.json'
  fold_input = {'name': jobname, 'sequences': chains, 'modelSeeds': seed_list,
                'dialect': 'alphafold3', 'version': 1}
  with open(json_path, 'w') as f:
    json.dump(fold_input, f, indent=2)
  jobs.append({'jobname': jobname, 'job_dir': job_dir, 'json_path': json_path, 'header': header})

print(f'Parsed {len(jobs)} job(s) from {fasta_path}')

skip_completed_jobs = True #@param {type:"boolean"}
#@markdown Drop entries that already have a finished prediction (checked in `af3_output/`
#@markdown and the Drive backup), so re-running this cell after an interrupted batch only
#@markdown launches the jobs still missing. `all_jobs` always keeps the full set, for the
#@markdown summary/display cells.

def is_job_done(job):
  jobname, job_dir = job['jobname'], job['job_dir']
  if os.path.exists(f'{job_dir}/{jobname}_summary_confidences.json'):
    return True
  return os.path.exists(f'{DRIVE_DIR}/{jobname}/{jobname}_summary_confidences.json')

all_jobs = jobs  # unfiltered - used by summary/display cells
if skip_completed_jobs:
  jobs = [j for j in all_jobs if not is_job_done(j)]
  print(f'{len(all_jobs) - len(jobs)}/{len(all_jobs)} already completed - skipped.')
else:
  jobs = all_jobs

print(f'{len(jobs)} job(s) to run:')
for j in jobs:
  print(' -', j['jobname'], '<-', j['header'])

In [ ]:
#@title Helpers: display, metrics, Drive backup
import csv, glob, json, os, shutil
import numpy as np

def collect_models(job_dir, jobname):
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

def load_pae(job_dir, jobname, cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json') if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

def display_job(job, load_as_frames=True, viewer_size=400):
  import py2Dmol
  job_dir, jobname = job['job_dir'], job['jobname']
  cifs = collect_models(job_dir, jobname)
  if not cifs:
    print(f'No model CIFs found in {job_dir}/'); return
  viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
  for i, cif in enumerate(cifs, start=1):
    pae = load_pae(job_dir, jobname, cif)
    name = 'models' if load_as_frames else f'rank_{i}'
    viewer.add_pdb(cif, name=name, paes=pae)
  viewer.show()

def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

def show_metrics(job):
  import matplotlib.pyplot as plt
  job_dir, jobname = job['job_dir'], job['jobname']
  conf_path = f'{job_dir}/{jobname}_confidences.json'
  summ_path = f'{job_dir}/{jobname}_summary_confidences.json'
  if not (os.path.exists(conf_path) and os.path.exists(summ_path)):
    print(f'No confidence files for {jobname} (job may have failed)'); return None
  with open(conf_path) as f: conf = json.load(f)
  with open(summ_path) as f: summ = json.load(f)

  plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
  plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))
  token_chain_ids = conf.get('token_chain_ids', [])
  pae = np.array(conf.get('pae', []), dtype=float)

  mean_plddt = summ.get('mean_plddt')
  if mean_plddt is None and plddts.size:
    mean_plddt = float(np.mean(plddts))
  iptm = summ.get('iptm')

  print('=' * 38)
  print(jobname)
  print(f'Mean pLDDT     : {fmt(mean_plddt)}')
  print(f'pTM            : {fmt(summ.get("ptm"))}')
  print(f'ipTM           : {fmt(iptm)}' + ('   (single chain)' if iptm is None else ''))
  print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
  print('=' * 38)

  has_pae = pae.ndim == 2 and pae.size > 0
  fig, axes = plt.subplots(1, 2 if has_pae else 1, figsize=(13 if has_pae else 6.5, 4))
  axes = np.atleast_1d(axes)

  ax = axes[0]
  x = np.arange(len(plddts)); xmax = max(len(plddts) - 1, 1)
  ax.set_xlim(0, xmax); ax.set_ylim(0, 100)
  unique_chains = list(dict.fromkeys(plddt_chain_ids))
  if len(unique_chains) > 1:
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
    tcid = np.array(plddt_chain_ids)
    for ch, col in zip(unique_chains, colors):
      y = np.where(tcid == ch, plddts, np.nan)
      ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
    for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
      ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
    ax.legend(loc='lower right', fontsize=8)
  else:
    ax.plot(x, plddts, lw=1.5, color='#1f77b4')
  for y in (50, 70, 90):
    ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
    ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
  ax.set_xlabel('Atom'); ax.set_ylabel('pLDDT'); ax.set_title(f'{jobname}: pLDDT')

  if has_pae:
    ax = axes[1]
    im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
    if token_chain_ids:
      for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
        ax.axhline(b - 0.5, c='black', lw=0.8)
        ax.axvline(b - 0.5, c='black', lw=0.8)
    ax.set_xlabel('Scored residue'); ax.set_ylabel('Aligned residue'); ax.set_title('PAE')

  plt.tight_layout(); plt.show()
  return {'jobname': jobname, 'header': job.get('header', ''), 'mean_plddt': mean_plddt,
          'ptm': summ.get('ptm'), 'iptm': iptm, 'ranking_score': summ.get('ranking_score')}

def sync_job_to_drive(job, drive_dir):
  dest = f'{drive_dir}/{job["jobname"]}'
  shutil.rmtree(dest, ignore_errors=True)
  shutil.copytree(job['job_dir'], dest)
  print(f'  -> synced to {dest}/')

In [ ]:
#@title Run AlphaFold 3
%%time
import os, shutil, subprocess, glob

#@markdown Inference settings (defaults match AlphaFold 3 - increase only if needed):
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes through the network (default 10). More can help large/hard targets, but is slower.
#@markdown - `num_diffusion_samples`: candidate structures generated per seed (default 5). Total models = seeds x samples.

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights: use the official AlphaFold 3 weights if they were fetched, else OpenFold3.
if os.path.isdir('af3_native_weights') and glob.glob('af3_native_weights/*.bin*'):
  model_dir, use_of3 = 'af3_native_weights', False
  print('Weights: official AlphaFold 3')
else:
  model_dir, use_of3 = 'af3_converted_weights', True
  print('Weights: OpenFold3')

# Here, the batch jobs are executed. For each job, we check if results already exist and either skip or run the AlphaFold 3 pipeline accordingly.
import csv

summary_rows = []

for j in jobs:
  jobname, job_dir, json_path = j['jobname'], j['job_dir'], j['json_path']

  have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
  if on_existing == 'skip' and have_results:
    print(f'Skipping {jobname}: results already exist in {job_dir}/')
  else:
    shutil.rmtree(job_dir, ignore_errors=True)

    cmd = [
        'python', 'run_alphafold.py',
        f'--json_path={json_path}',
        f'--model_dir={model_dir}',
        '--norun_data_pipeline',
        f'--output_dir={OUTPUT_DIR}',
        '--cache_dir=/tmp/af3_cache',
        '--force_output_dir',
        f'--flash_attention_implementation={flash_impl}',
        f'--num_recycles={num_recycles}',
        f'--num_diffusion_samples={num_diffusion_samples}',
    ]
    if msa_mode == 'mmseqs2_server':
      cmd.append('--use_msa_server')
    if nojit:
      cmd.append('--nojit')
    if use_of3:
      cmd.append('--of3_weights')

    cmd = ' '.join(cmd)
    print(f'\n=== {jobname} ===')
    print(cmd)
    !{cmd}
    print(f'Done -> {job_dir}/')

  # Show results and back up immediately, so a later interrupt can't lose this job
  display_job(j)
  row = show_metrics(j)
  if row:
    summary_rows.append(row)
  sync_job_to_drive(j, DRIVE_DIR)

if summary_rows:
  summary_csv = f'{OUTPUT_DIR}/batch_summary.csv'
  fields = ['jobname', 'header', 'mean_plddt', 'ptm', 'iptm', 'ranking_score']
  with open(summary_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(sorted(summary_rows, key=lambda r: -(r['ranking_score'] or 0)))
  shutil.copy(summary_csv, f'{DRIVE_DIR}/batch_summary.csv')
  print(f'\nBatch summary -> {summary_csv} (synced to Drive)')


In [ ]:
#@title Build/update batch_summary.csv (post-hoc, safe to re-run anytime)
import csv, json, os, shutil

def resolve_result_dir(job):
  """Local af3_output/ if the job finished this session, else the Drive backup."""
  local_dir = job['job_dir']
  if os.path.exists(f'{local_dir}/{job["jobname"]}_summary_confidences.json'):
    return local_dir
  drive_dir = f'{DRIVE_DIR}/{job["jobname"]}'
  if os.path.exists(f'{drive_dir}/{job["jobname"]}_summary_confidences.json'):
    return drive_dir
  return None

summary_rows = []
for j in all_jobs:
  jobname = j['jobname']
  result_dir = resolve_result_dir(j)
  if result_dir is None:
    continue  # not finished (or failed) yet - re-run this cell later to pick it up

  with open(f'{result_dir}/{jobname}_summary_confidences.json') as f:
    summ = json.load(f)

  mean_plddt = summ.get('mean_plddt')
  conf_path = f'{result_dir}/{jobname}_confidences.json'
  if mean_plddt is None and os.path.exists(conf_path):
    with open(conf_path) as f:
      conf = json.load(f)
    plddts = conf.get('atom_plddts', conf.get('token_plddts', []))
    if plddts:
      mean_plddt = sum(plddts) / len(plddts)

  summary_rows.append({
      'jobname': jobname, 'header': j.get('header', ''),
      'mean_plddt': mean_plddt, 'ptm': summ.get('ptm'),
      'iptm': summ.get('iptm'), 'ranking_score': summ.get('ranking_score'),
  })

summary_csv = f'{OUTPUT_DIR}/batch_summary.csv'
fields = ['jobname', 'header', 'mean_plddt', 'ptm', 'iptm', 'ranking_score']
with open(summary_csv, 'w', newline='') as f:
  w = csv.DictWriter(f, fieldnames=fields)
  w.writeheader()
  w.writerows(sorted(summary_rows, key=lambda r: -(r['ranking_score'] or 0)))
shutil.copy(summary_csv, f'{DRIVE_DIR}/batch_summary.csv')

print(f'{len(summary_rows)}/{len(all_jobs)} jobs finished so far -> {summary_csv} (synced to Drive)')

In [ ]:
#@title Parse per-sample confidences into overview_per-sample.csv
import csv, glob, json, os, re, shutil

def parse_all_samples():
  rows = []
  for j in all_jobs:
    jobname, job_dir, header = j['jobname'], j['job_dir'], j.get('header', '')
    for path in sorted(glob.glob(f'{job_dir}/seed-*_sample-*/*_summary_confidences.json')):
      d = os.path.basename(os.path.dirname(path))  # 'seed-<n>_sample-<m>'
      m = re.match(r'seed-(\d+)_sample-(\d+)$', d)
      if not m:
        continue
      seed, sample = int(m.group(1)), int(m.group(2))
      with open(path) as f:
        summ = json.load(f)

      # chain_ids is per-token/atom (repeated); de-dup keeping first-seen order
      unique_chains = list(dict.fromkeys(summ.get('chain_ids', [])))
      chain_ptm = summ.get('chain_ptm', [])
      chain_iptm = summ.get('chain_iptm', [])

      row = {
          'jobname': jobname,
          'header': header,
          'seed': seed,
          'sample': sample,
          'ptm': summ.get('ptm'),
          'iptm': summ.get('iptm'),
          'ranking_score': summ.get('ranking_score'),
          'fraction_disordered': summ.get('fraction_disordered'),
          'has_clash': summ.get('has_clash'),
          'chain_ids': ','.join(unique_chains),
          'chain_pair_iptm': json.dumps(summ.get('chain_pair_iptm')),
          'chain_pair_pae_min': json.dumps(summ.get('chain_pair_pae_min')),
      }
      for ch, v in zip(unique_chains, chain_ptm):
        row[f'ptm_chain_{ch}'] = v
      for ch, v in zip(unique_chains, chain_iptm):
        row[f'iptm_chain_{ch}'] = v
      rows.append(row)
  return rows

rows = parse_all_samples()

fixed_cols = ['jobname', 'header', 'seed', 'sample', 'ptm', 'iptm', 'ranking_score',
              'fraction_disordered', 'has_clash', 'chain_ids']
# Per-chain columns vary in count across jobs (protein-only vs. +ligand chains)
chain_cols = sorted({k for r in rows for k in r if k.startswith(('ptm_chain_', 'iptm_chain_'))})
extra_cols = ['chain_pair_iptm', 'chain_pair_pae_min']
fieldnames = fixed_cols + chain_cols + extra_cols

out_path = f'{OUTPUT_DIR}/overview_per-sample.csv'
with open(out_path, 'w', newline='') as f:
  w = csv.DictWriter(f, fieldnames=fieldnames)
  w.writeheader()
  w.writerows(sorted(rows, key=lambda r: (r['jobname'], r['seed'], r['sample'])))

print(f'Wrote {len(rows)} sample rows ({len(all_jobs)} jobs) -> {out_path}')
shutil.copy(out_path, f'{DRIVE_DIR}/overview_per-sample.csv')
print(f'Synced to {DRIVE_DIR}/overview_per-sample.csv')

In [ ]:
#@title Display structures + PAE (re-view a specific job)
job_index = 0 #@param {type:"integer"}
load_as_frames = True #@param {type:"boolean"}
display_job(all_jobs[job_index], load_as_frames=load_as_frames)

In [ ]:
#@title Quality metrics and plots (re-view a specific job)
job_index = 0 #@param {type:"integer"}
show_metrics(all_jobs[job_index])

In [ ]:
#@title Download results (local zip, optional — Drive already has a live backup)
from google.colab import files
import os

results_zip = 'af3_batch_results.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Fill in the sequence(s) in the **Input sequence(s)** cell.
2. Press **Runtime → Run all**.
3. The install cell (first run only) downloads OpenFold3 weights and builds AF3 data files in the background — subsequent runs reuse them.

---

## Model weights

By default the notebook uses the open **OpenFold3** weights (Apache-2.0, no restrictions). To run the **official AlphaFold 3** weights instead, tick the **use_af3_weights** box in the install cell. The weights are now publicly downloadable (no login, key, or DeepMind approval required) and are fetched directly into `af3_native_weights/`, then used automatically (the run cell detects them and drops the `--of3_weights` flag). Leaving **use_af3_weights** off keeps the OpenFold3 path.

Official AlphaFold 3 parameters are subject to the [AF3 weights terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md); run_alphafold prints a reminder of these terms when they are used.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | Apache 2.0 notice (OpenFold3 weights have no commercial restrictions). |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the OF3 checkpoint is ~1.4 GB from AWS S3; the install cell waits automatically.

## License

AlphaFold 3 source code and OpenFold3 weights are both licensed under [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Outputs generated with OpenFold3 weights are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of Use and may be used freely, including commercially.

## Bugs / feedback

Report issues at https://github.com/sokrypton/alphafold3/issues